In [14]:
from transformers import AutoTokenizer

# Load a pre-trained tokenizer
model_name = "meta-llama/Meta-Llama-3-8B"  # deepseek-ai/DeepSeek-V2"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize a text
text = "The quick brown fox jumps over the lazy dog!"

encoded_text = tokenizer(text)

# Access the tokenized IDs
print(tokenizer.decode(encoded_text["input_ids"]))

print(encoded_text["input_ids"])
print(", ".join([tokenizer.decode(t) for t in encoded_text["input_ids"]]))

[CLS] the quick brown fox jumps over the lazy dog! [SEP]
[101, 1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899, 999, 102]
[CLS], the, quick, brown, fox, jumps, over, the, lazy, dog, !, [SEP]


In [15]:
import os
os.environ["DEEPSEEK_API_KEY"] = "sk-42e0f925f1d44cd28093b459f47fe214"
from langchain_deepseek import ChatDeepSeek
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
llm = ChatDeepSeek(model="deepseek-v4-flash")
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
# llm = ChatOpenAI(model="gpt-4")

import math

def mocked_google_search(query: str) -> str:
  print(f"CALLED GOOGLE_SEARCH with query={query}")
  return "Donald Trump is a president of USA and he's 78 years old"

def mocked_calculator(expression: str) -> float:
  print(f"CALLED CALCULATOR with expression={expression}")
  if "sqrt" in expression:
    return math.sqrt(78*132)
  return 78*132

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Computes mathematical expressions",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "title": "expression",
                    "description": "A mathematical expression to be evaluated by a calculator"
                }
            },
            "required": ["expression"]
        }
    }
}

search_tool = {
    "type": "function",
    "function": {
        "name": "google_search",
        "description": "Returns about common facts, fresh events and news from Google Search engine based on a query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "title": "search_query",
                    "description": "Search query to be sent to the search engine"
                }
            },
            "required": ["query"]
        }
    }
}

system_prompt = (
    "Always use a calculator for mathematical computations, and use Google Search "
    "for information about common facts, fresh events and news. Do not assume anything, keep in "
    "mind that things are changing and always "
    "check yourself with external sources if possible."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="messages"),
])

llm_with_tools = prompt | llm.bind_tools([search_tool, calculator_tool])

from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import MessagesState, StateGraph, START, END


def invoke_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


def call_tools(state: MessagesState):
    last_message = state["messages"][-1]
    tool_calls = last_message.tool_calls

    new_messages = []

    for tool_call in tool_calls:
      if tool_call["name"] == "google_search":
        tool_result = mocked_google_search(**tool_call["args"])
        new_messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))
      elif tool_call["name"] == "calculator":
        tool_result = mocked_calculator(**tool_call["args"])
        new_messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))
      else:
        raise ValueError(f"Tool {tool_call['name']} is not defined!")
    return {"messages": new_messages}


def should_run_tools(state: MessagesState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
      return "call_tools"
    return END

builder = StateGraph(MessagesState)
builder.add_node("invoke_llm", invoke_llm)
builder.add_node("call_tools", call_tools)

builder.add_edge(START, "invoke_llm")
builder.add_conditional_edges("invoke_llm", should_run_tools)
builder.add_edge("call_tools", "invoke_llm")
graph = builder.compile()

question = "What is a square root of the current US president’s age multiplied by 132?"
result = graph.invoke({"messages": [HumanMessage(content=question)]})
print(result["messages"][-1].content)


CALLED CALCULATOR with expression=sqrt(78*132)
The current U.S. president is **Donald Trump**, who is **78 years old** as of June 9, 2025.

\[
\sqrt{78 \times 132}=\sqrt{10{,}296}\approx \boxed{101.47}
\]

So the square root is approximately **101.47**.


In [34]:
import os
os.environ["DEEPSEEK_API_KEY"] = "sk-42e0f925f1d44cd28093b459f47fe214"
from langchain_deepseek import ChatDeepSeek
from langchain_community.tools import DuckDuckGoSearchRun
llm = ChatDeepSeek(model="deepseek-v4-flash")
search = DuckDuckGoSearchRun(api_wrapper_kwargs={"backend": "api"})
print(f"Tool's name = {search.name}")
print(f"Tool's name = {search.description}")
print(f"Tool's arg schema = {search.args_schema}")
from langchain_community.tools.ddg_search.tool import DDGInput
print(DDGInput.model_fields)
query = "What is the weather in Munich like tomorrow?"
search_input = DDGInput(query=query)
result = search.invoke(search_input.model_dump())
print(result)
llm_with_tools = llm.bind_tools([search])
result = llm_with_tools.invoke([("system", "Always use a duckduckgo_search tool for queries that require a fresh information"), ("user", query)])
print(result.tool_calls[0])

Tool's name = duckduckgo_search
Tool's name = A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
Tool's arg schema = <class 'langchain_community.tools.ddg_search.tool.DDGInput'>
{'query': FieldInfo(annotation=str, required=True, description='search query to look up')}
Today's and tonight's professional weather forecast for Munich. Precipitation radar, HD satellite images, and current weather warnings, hourly temperature, chance of rain, and sunshine hours. Nov 12, 2026 · Everything you need to know about tomorrow's weather in Munich, Bavaria, Germany. High/Low, Precipitation Chances, Sunrise/Sunset, and tomorrow's Temperature History. Nov 12, 2026 · Munich weather tomorrow, hour by hour: temperature, chance of rain, wind, UV and air quality, plus sunrise and sunset times. 3 days ago · Munich, Germany - Detailed weather forecast for tomorrow. Hourly forecast for tomorrow - including weather conditions, t